<div style="font-size:30px;font-weight:700;color:#111827;padding-bottom:8px;margin:18px 0;">
자동 미분(autograd)
</div>

# leaf 텐서 설정 (requires_grad=True 는 가중치에만)

**목표**: 학습 대상 가중치 `w` / `b` 에 `requires_grad=True` 를 붙이고, 입력 `x` / 타깃 `y` 는 default(False) 그대로 둔다.

In [1]:
import torch

torch.manual_seed(0)

x = torch.ones(5)                                # 입력
y = torch.zeros(3)                               # 타깃 (binary)
w = torch.randn(5, 3, requires_grad=True)        # 학습 대상, 미분을 수행하려면 반드시 requires_grad가 ture로 설정
b = torch.randn(3, requires_grad=True)           # 학습 대상

print(f"w.shape = {tuple(w.shape)}, w.is_leaf = {w.is_leaf}, "
      f"w.requires_grad = {w.requires_grad}")
print(f"b.shape = {tuple(b.shape)}, b.is_leaf = {b.is_leaf}, "
      f"b.requires_grad = {b.requires_grad}")
print(f"x.shape = {tuple(x.shape)}, x.requires_grad = {x.requires_grad}")
print(f"y.shape = {tuple(y.shape)}, y.requires_grad = {y.requires_grad}")

w.shape = (5, 3), w.is_leaf = True, w.requires_grad = True
b.shape = (3,), b.is_leaf = True, b.requires_grad = True
x.shape = (5,), x.requires_grad = False
y.shape = (3,), y.requires_grad = False


**해석**: `w` 와 `b` 만 leaf + requires_grad — 다음 단계의 forward 가 두 텐서에 적용되는 모든 연산을 그래프에 기록합니다. `x` 와 `y` 의 default False 는 의도적 — 입력에 grad 가 추적되면 메모리·시간이 낭비됩니다.


# forward 식 작성 + grad_fn 확인

**목표**: 식 한 줄로 그래프 노드 하나가 자동 추가되고, 결과 텐서에 `grad_fn` 이 달려 있음을 출력으로 확인한다.

In [2]:
z = torch.matmul(x, w) + b
loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)

print(f"z.device      = {z.device}")
print(f"z.shape       = {tuple(z.shape)}")
print(f"z.grad_fn     = {z.grad_fn}")
print(f"loss.grad_fn  = {loss.grad_fn}")
print(f"loss.item()   = {loss.item():.4f}")
print(f"x.grad_fn     = {x.grad_fn}")        # leaf 는 None

z.device      = cpu
z.shape       = (3,)
z.grad_fn     = <AddBackward0 object at 0x7cee783ed8a0>
loss.grad_fn  = <BinaryCrossEntropyWithLogitsBackward0 object at 0x7cee784d1990>
loss.item()   = 0.6819
x.grad_fn     = None


# `loss.backward()` 한 줄로 leaf .grad 채우기

**목표**: backward 호출 한 줄로 `w.grad`, `b.grad` 가 채워지고, `x.grad` 는 여전히 None 임을 확인한다.

In [3]:
loss.backward()

print(f"w.grad = {w.grad}")
print(f"w.grad.shape  = {tuple(w.grad.shape)}")
print(f"b.grad        = {b.grad}")
print(f"x.grad        = {x.grad}")            # default False 라 None

w.grad = tensor([[0.2661, 0.1179, 0.0027],
        [0.2661, 0.1179, 0.0027],
        [0.2661, 0.1179, 0.0027],
        [0.2661, 0.1179, 0.0027],
        [0.2661, 0.1179, 0.0027]])
w.grad.shape  = (5, 3)
b.grad        = tensor([0.2661, 0.1179, 0.0027])
x.grad        = None


# 두 번째 backward 호출로 누적 검증

**목표**: 새 forward 빌드 + 두 번째 backward 호출 후 `w.grad` 가 첫 결과의 약 2 배가 되어 있음을 확인한다.

In [4]:
prev_grad_first_row = w.grad[0].clone()          # 첫 결과 보관
print(f"1st backward 후 w.grad[0] = {prev_grad_first_row}")

# 같은 그래프는 free 됐으므로 새 forward 빌드
z2 = torch.matmul(x, w) + b
loss2 = torch.nn.functional.binary_cross_entropy_with_logits(z2, y)
loss2.backward()

print(f"2nd backward 후 w.grad[0] = {w.grad[0]}")
print(f"두 배 검증 — torch.allclose(2*first, second) = "
      f"{torch.allclose(2 * prev_grad_first_row, w.grad[0])}") # 두 텐서의 값이 거의 같은지 비교하는 함수

1st backward 후 w.grad[0] = tensor([0.2661, 0.1179, 0.0027])
2nd backward 후 w.grad[0] = tensor([0.5322, 0.2359, 0.0054])
두 배 검증 — torch.allclose(2*first, second) = True


# `grad.zero_()` 후 단일 호출 값으로 복귀

**목표**: in-place `zero_()` 로 `.grad` 를 비우고 다시 backward 하면 값이 첫 호출과 같아짐을 확인한다.

**코드**:


In [5]:
w.grad.zero_() #w에 미분된 값을 초기화
b.grad.zero_() #b에 미분된 값을 초기화

z3 = torch.matmul(x, w) + b
loss3 = torch.nn.functional.binary_cross_entropy_with_logits(z3, y)
loss3.backward()

print(f"zero 후 backward — w.grad[0] = {w.grad[0]}")
print(f"단일 호출 값과 일치 — torch.allclose(first, zeroed) = "
      f"{torch.allclose(prev_grad_first_row, w.grad[0])}")

zero 후 backward — w.grad[0] = tensor([0.2661, 0.1179, 0.0027])
단일 호출 값과 일치 — torch.allclose(first, zeroed) = True


# `torch.no_grad()` 컨텍스트로 추론 wrapper

**목표**: 같은 forward 식을 `with torch.no_grad():` 블록 안에서 실행했을 때 결과 텐서의 `requires_grad` 가 False 임을 확인한다.

In [7]:
z_train = torch.matmul(x, w) + b
print(f"학습 경로 — z_train.requires_grad = {z_train.requires_grad}")

with torch.no_grad():
    z_infer = torch.matmul(x, w) + b #a() 안에 들어가 있어서 미분을 안함, 학습이 아닌 경우 이런 방법으로 수행
print(f"추론 경로 — z_infer.requires_grad = {z_infer.requires_grad}")
print(f"추론 경로 — z_infer.grad_fn        = {z_infer.grad_fn}")

학습 경로 — z_train.requires_grad = True
추론 경로 — z_infer.requires_grad = False
추론 경로 — z_infer.grad_fn        = None


# `tensor.detach()` 로 단일 텐서를 그래프에서 떼내기

**목표**: 학습 경로에서 만든 텐서를 `.detach()` 호출 한 번으로 그래프에서 분리한다.

**코드**:


In [8]:
z = torch.matmul(x, w) + b
print(f"detach 전 — z.requires_grad     = {z.requires_grad}")
print(f"detach 전 — z.grad_fn           = {z.grad_fn}")

z_det = z.detach()
print(f"detach 후 — z_det.requires_grad = {z_det.requires_grad}")
print(f"detach 후 — z_det.grad_fn       = {z_det.grad_fn}")

detach 전 — z.requires_grad     = True
detach 전 — z.grad_fn           = <AddBackward0 object at 0x7cee7813fd90>
detach 후 — z_det.requires_grad = False
detach 후 — z_det.grad_fn       = None


# 단일 파라미터 동결: `requires_grad_(False)`

**목표**: 학습 중인 leaf 텐서에 in-place 로 `requires_grad_(False)` 를 적용해 그 텐서가 그래프에서 제외됨을 확인한다.

In [9]:
w_a = torch.randn(3, 3, requires_grad=True)
w_b = torch.randn(3, 3, requires_grad=True)
x_dummy = torch.ones(3)
target = torch.zeros(3)

# w_b 만 동결
w_b.requires_grad_(False) #중간에 변경한거라 b만 미분을 안 수행

loss_freeze = (w_a @ x_dummy + w_b @ x_dummy - target).pow(2).sum()
loss_freeze.backward()

print(f"w_a.grad is not None      = {w_a.grad is not None}")
print(f"w_a.grad.shape            = {tuple(w_a.grad.shape)}")
print(f"w_b.grad is None          = {w_b.grad is None}")    # 동결됐으니 None
print(f"w_b.requires_grad         = {w_b.requires_grad}")    # False

w_a.grad is not None      = True
w_a.grad.shape            = (3, 3)
w_b.grad is None          = True
w_b.requires_grad         = False
